# 05 · Uçtan Uca Sistem: Belge → Tür → İçerik

```
Belge (resim / PDF) → OCR → Metin temizleme → Tür tahmini → Türe özel içerik çıkarımı → "Bu bir CV: ..."
```

| Hocanın maddesi | Bölüm |
|---|---|
| Final 5 · Yeni belge → OCR → tür tahmini | 4–6 |
| İlk istek · Belgenin içeriğini sunmak (ör. CV ise ad, üniversite, bölüm) | 3–6 |
| İçerik çıkarımının değerlendirilmesi | 7 |
| Demo arayüzü | 8 |

**Girdi:** `models/best_model.joblib` (04) · Önce 04 çalışmış olmalı. GPU gerekmez.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null
!pip -q install pytesseract opencv-python-headless pdf2image spacy gradio
!python -m spacy download en_core_web_sm -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, joblib
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
MODELS = ROOT / "models"
OWN = ROOT / "kendi_belgelerim"; OWN.mkdir(exist_ok=True)   # kendi belgelerinizi buraya koyun
CLASSES = ["email", "form", "invoice", "resume"]
SEED = 42
sys.path.insert(0, str(ROOT))

## 2. İçerik çıkarım modülü

Belge türü belirlendikten sonra, o türe özel alanlar iki yöntemle çıkarılır:

| Yöntem | Kullanıldığı yer |
|---|---|
| **Kurallar (regex)** | E-posta adresi, telefon, tarih, tutar, fatura no, `From:` / `Subject:` başlıkları, `Alan: değer` çiftleri |
| **İsim tanıma (spaCy NER)** | Kişi adı (CV), şirket adı (fatura) |
| **Bölüm başlıkları** | CV'de `EDUCATION` altındaki satırlardan üniversite ve bölüm, `SKILLS` altından beceriler |

| Tür | Çıkarılan alanlar |
|---|---|
| CV | ad-soyad, üniversite, bölüm/derece, e-posta, telefon, beceriler |
| Fatura | fatura no, tarih, firma, toplam tutar |
| E-posta | gönderen, alıcı, konu, tarih, gövde özeti |
| Form | başlık, alan–değer çiftleri, tarih |

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/extract.py
"""Belge türüne özel içerik çıkarımı (kural tabanlı + spaCy NER).

Girdi: OCR'dan gelen temizlenmiş metin (text_clean.clean_text çıktısı) ve tahmin edilen tür.
Çıktı: {"alanlar": {...}, "ozet": "Bu bir CV: ..."}
"""
import re

try:
    import spacy
    _NLP = spacy.load("en_core_web_sm")
except Exception:  # spaCy yoksa sadece kurallar çalışır
    _NLP = None

# ---------------------------------------------------------------- ortak kalıplar
EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
PHONE = re.compile(r"(?:\+?\d{1,3}[\s.-]?)?(?:\(\d{2,4}\)|\d{2,4})[\s.-]?\d{3}[\s.-]?\d{2,4}(?:[\s.-]?\d{2})?")
URL = re.compile(r"\b(?:https?://|www\.)\S+|\b(?:github|linkedin)\.com/\S+", re.I)
MONTHS = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]*\.?"
DATE = re.compile(
    rf"\b(?:\d{{1,2}}[/.-]\d{{1,2}}[/.-]\d{{2,4}}|{MONTHS}\s+\d{{1,2}},?\s+\d{{2,4}}|\d{{1,2}}\s+{MONTHS}\s+\d{{2,4}}|\d{{4}}-\d{{2}}-\d{{2}})\b",
    re.I)
MONEY = re.compile(r"(?:[$€£₺]\s?\d[\d,]*(?:\.\d{2})?|\d[\d,]*\.\d{2}\s?(?:usd|eur|tl|try)?)", re.I)

DEGREE_WORDS = ("bachelor", "master", "phd", "ph.d", "b.s", "b.sc", "m.s", "m.sc", "b.a", "m.a", "mba",
                "degree", "engineering", "science", "lisans", "mühendisliği", "muhendisligi")
SCHOOL_WORDS = ("university", "universit", "college", "institute", "school", "academy", "üniversite", "universite")
SECTION_HEADS = ("education", "experience", "work experience", "employment", "skills", "technical skills",
                 "projects", "references", "publications", "certifications", "languages", "interests",
                 "objective", "summary", "awards", "activities")


def _lines(text):
    return [l.strip() for l in text.splitlines() if l.strip()]


def _ents(text, labels):
    if _NLP is None or not text:
        return []
    doc = _NLP(text[:20000])
    out = []
    for e in doc.ents:
        t = " ".join(e.text.split())
        if e.label_ in labels and len(t) > 1 and t not in out:
            out.append(t)
    return out


def _first(pattern, text, group=0):
    m = pattern.search(text)
    return m.group(group).strip() if m else None


def _section(lines, names):
    """Başlığı `names` içinde olan bölümün satırlarını döndürür (bir sonraki başlığa kadar)."""
    out, inside = [], False
    for l in lines:
        key = re.sub(r"[^a-z ]", "", l.lower()).strip()
        is_head = key in SECTION_HEADS and len(l) < 40
        if is_head:
            if inside:
                break
            inside = key in names
            continue
        if inside:
            out.append(l)
    return out


_NOT_NAME = SECTION_HEADS + SCHOOL_WORDS + DEGREE_WORDS + ("resume", "curriculum", "vitae", "cv", "page",
                                                           "street", "avenue", "road", "phone", "email", "address")


def _looks_like_name(line):
    """2-4 kelimelik, harflerden oluşan, büyük harfle başlayan, başlık/okul/adres olmayan satır."""
    if any(ch.isdigit() for ch in line) or any(ch in line for ch in "@:/|()"):
        return False
    words = line.replace(",", " ").replace(".", " ").split()
    if not 2 <= len(words) <= 4 or not all(w.isalpha() for w in words) or sum(len(w) >= 2 for w in words) < 2:
        return False
    low = line.lower()
    if any(k in low.split() or k in low for k in _NOT_NAME if len(k) > 3) or low in _NOT_NAME:
        return False
    return all(w[0].isupper() for w in words)


# ---------------------------------------------------------------- türe özel çıkarıcılar
def extract_resume(text):
    lines = _lines(text)
    persons = [p for p in _ents("\n".join(lines[:8]), {"PERSON"}) if _looks_like_name(p)]
    name = persons[0] if persons else next((l for l in lines[:6] if _looks_like_name(l)), None)

    edu = _section(lines, {"education"}) or lines
    school = next((l for l in edu if any(w in l.lower() for w in SCHOOL_WORDS)), None)
    if school is None:
        orgs = [o for o in _ents("\n".join(edu), {"ORG"}) if any(w in o.lower() for w in SCHOOL_WORDS)]
        school = orgs[0] if orgs else None
    degree = next((l for l in edu if any(w in l.lower() for w in DEGREE_WORDS) and l != school), None)
    if school:
        school = re.sub(r"^\s*education\s*[:;-]\s*", "", school, flags=re.I) or None

    skills_lines = _section(lines, {"skills", "technical skills"})
    skills = None
    if skills_lines:
        items = re.split(r"[,;•·|]|\s-\s", " ".join(skills_lines[:4]))
        skills = [s.strip(" :.") for s in items if 1 < len(s.strip()) < 30][:10] or None

    return {
        "ad_soyad": name,
        "universite": school,
        "bolum_derece": degree,
        "e_posta": _first(EMAIL, text),
        "telefon": _first(PHONE, text),
        "beceriler": skills,
    }


_NOT_COMPANY = ("invoice", "quantity", "description", "price", "amount", "total", "date", "bill to",
                "ship to", "page", "tax", "qty", "item")
_CORP = re.compile(r"\b(?:inc|corp|corporation|co|company|ltd|llc|plc|group|gmbh|a\.?ş|ltd\.?\s*şti)\b\.?", re.I)


def _invoice_company(lines):
    top = [l for l in lines[:12] if not any(w in l.lower() for w in _NOT_COMPANY) and re.search(r"[A-Za-z]{3}", l)]
    corp = next((l for l in top if _CORP.search(l)), None)
    if corp:
        return corp
    orgs = [o for o in _ents("\n".join(top), {"ORG"}) if not any(w in o.lower() for w in _NOT_COMPANY)]
    if orgs:
        return orgs[0]
    # antetli kâğıt: ilk 3 satırdan büyük harf ağırlıklı olan
    def upper_ratio(l):
        letters = [c for c in l if c.isalpha()]
        return sum(c.isupper() for c in letters) / len(letters) if letters else 0
    return next((l for l in top[:3] if upper_ratio(l) > 0.6 and len(l) > 4), None)


def extract_invoice(text):
    text = re.sub(r"(\d)[,.]\s+(\d{3})\b", r"\1,\2", text)   # OCR: '1, 250.00' -> '1,250.00'
    lines = _lines(text)
    inv_no = _first(re.compile(r"invoice\s*(?:no\.?|number|num|#)\s*[:.]?\s*((?=[A-Z0-9/-]*\d)[A-Z0-9][A-Z0-9/-]{1,20})", re.I), text, 1)
    total = None
    for pat in (r"(?:total\s+amount\s+due|amount\s+due|balance\s+due|grand\s+total|total\s+due|invoice\s+total|total)\b[^\n$\d]{0,20}([$€£]?\s?\d[\d,]*(?:\.\d{2})?)",
                r"\bamount\b[^\n$\d]{0,15}([$€£]?\s?\d[\d,]*(?:\.\d{2})?)"):
        m = re.findall(pat, text, re.I)
        m = [x.strip().rstrip(",.") for x in m if re.search(r"\d\.\d{2}$|\d{3,}", x.strip().rstrip(",."))]
        if m:
            total = m[-1]
            break
    company = _invoice_company(lines)
    return {
        "fatura_no": inv_no,
        "tarih": _first(DATE, text),
        "firma": company,
        "toplam_tutar": total,
        "tutarlar": MONEY.findall(text)[:5] or None,
    }


def _header(text, names):
    pat = re.compile(rf"^\s*(?:{names})\s*[:;]\s*(.+)$", re.I | re.M)
    m = pat.search(text)
    return m.group(1).strip() if m else None


def extract_email(text):
    lines = _lines(text)
    sender = _header(text, "from|sender")
    to = _header(text, "to|recipient")
    subject = _header(text, "subject|re|subj")
    date = _header(text, "date|sent") or _first(DATE, text)
    head_pat = re.compile(r"^\s*(?:from|to|cc|bcc|subject|subj|re|date|sent|sender|recipient)\s*[:;]", re.I)
    body = [l for l in lines if not head_pat.match(l)]
    return {
        "gonderen": sender,
        "alici": to,
        "konu": subject,
        "tarih": date,
        "e_postalar": EMAIL.findall(text)[:5] or None,
        "govde_ozeti": " ".join(body[:3])[:200] or None,
    }


def extract_form(text):
    lines = _lines(text)
    title = next((l for l in lines[:8] if l.isupper() and len(l.split()) >= 2), lines[0] if lines else None)
    pairs = {}
    for l in lines:
        m = re.match(r"^([A-Za-z][A-Za-z ./&()'-]{1,40}?)\s*[:;]\s*(.+)$", l)
        if m:
            key, val = m.group(1).strip(), m.group(2).strip(" _.")
            if val and key.lower() not in pairs and len(val) < 120:
                pairs[key.lower()] = val
        if len(pairs) >= 10:
            break
    return {
        "baslik": title,
        "alanlar": pairs or None,
        "tarih": _first(DATE, text),
    }


EXTRACTORS = {"resume": extract_resume, "invoice": extract_invoice, "email": extract_email, "form": extract_form}
TR_NAMES = {"resume": "CV", "invoice": "fatura", "email": "e-posta", "form": "form"}


def summarize(label, fields):
    f = fields
    if label == "resume":
        parts = [f.get("ad_soyad"), f.get("universite"), f.get("bolum_derece")]
        if f.get("beceriler"):
            parts.append("Beceriler: " + ", ".join(f["beceriler"][:5]))
    elif label == "invoice":
        parts = [f"Fatura no {f['fatura_no']}" if f.get("fatura_no") else None, f.get("firma"), f.get("tarih"),
                 f"Toplam {f['toplam_tutar']}" if f.get("toplam_tutar") else None]
    elif label == "email":
        who = " → ".join(x for x in (f.get("gonderen"), f.get("alici")) if x) or None
        parts = [who, f"Konu: {f['konu']}" if f.get("konu") else None, f.get("tarih")]
    else:
        kv = list((f.get("alanlar") or {}).items())[:4]
        parts = [f.get("baslik")] + [f"{k} = {v}" for k, v in kv]
    parts = [p for p in parts if p]
    return f"Bu bir {TR_NAMES[label]}" + (": " + " · ".join(parts) if parts else " (içerik alanları okunamadı)")


def extract(text, label):
    fields = EXTRACTORS[label](text or "")
    return {"alanlar": fields, "ozet": summarize(label, fields)}


In [ ]:
import ocr_core, text_clean, extract
for m in (ocr_core, text_clean, extract): importlib.reload(m)
from ocr_core import run_ocr
from text_clean import clean_text, normalize_for_model
from extract import extract as extract_fields, TR_NAMES
print("spaCy yüklü:", extract._NLP is not None)

## 3. Model: olasılık veren sürüm

04'teki en iyi model (Linear SVM) olasılık vermez, sadece karar skoru verir. Demoda "%95 emin" diyebilmek için aynı model
**olasılık kalibrasyonu** (Platt yöntemi, 5 katlı) ile train verisinde yeniden eğitilir. Tahminler aynı kalır; test başarısı aşağıda kontrol edilir.

In [ ]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score

base = joblib.load(MODELS / "best_model.joblib")
info = json.loads((MODELS / "best_model.json").read_text())
df = pd.read_csv(DATA / "cleaned_dataset.csv"); df["text_norm"] = df["text_norm"].fillna("")
train = df[(df.split == "train") & (~df.is_empty)]
test = df[df.split == "test"]

model = CalibratedClassifierCV(clone(base), cv=5, method="sigmoid").fit(train.text_norm, train.label)
for name, m in [("04 modeli", base), ("kalibre model", model)]:
    p = m.predict(test.text_norm)
    print(f"{name:14s} test accuracy {accuracy_score(test.label, p):.4f} · macro-F1 {f1_score(test.label, p, average='macro'):.4f}")
joblib.dump(model, MODELS / "demo_model.joblib")
print("Model:", info["model"], "·", info["özellik"], "TF-IDF")

## 4. Uçtan uca fonksiyon

In [ ]:
from pdf2image import convert_from_path

def load_document(path):
    """Resim veya PDF (ilk sayfa) → gri tonlamalı görüntü."""
    path = str(path)
    if path.lower().endswith(".pdf"):
        page = convert_from_path(path, dpi=200, first_page=1, last_page=1)[0]
        return cv2.cvtColor(np.array(page), cv2.COLOR_RGB2GRAY)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Okunamadı: {path}")
    return img

def analyze(img, min_conf=0.5):
    """Görüntü → OCR → temizleme → tür tahmini → içerik çıkarımı."""
    ocr = run_ocr(img)                                   # v2: ham görüntü (pilot deney sonucu)
    clean = clean_text(ocr["text"])
    if len(clean.split()) < 5:
        return {"tur": None, "olasiliklar": {}, "ozet": "Belgeden yeterli metin okunamadı.",
                "alanlar": {}, "ocr_metni": ocr["text"], "ocr_guveni": ocr["mean_conf"]}
    proba = model.predict_proba([normalize_for_model(clean)])[0]
    probs = dict(zip(model.classes_, proba.round(4)))
    label = max(probs, key=probs.get)
    ext = extract_fields(clean, label)
    note = "" if probs[label] >= min_conf else f" (model emin değil, %{100 * probs[label]:.0f})"
    return {"tur": label, "olasiliklar": probs, "ozet": ext["ozet"] + note,
            "alanlar": ext["alanlar"], "ocr_metni": clean, "ocr_guveni": round(ocr["mean_conf"], 1)}

def show(path, res, true_label=None):
    fig, ax = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={"width_ratios": [1, 1.2]})
    ax[0].imshow(load_document(path), cmap="gray"); ax[0].axis("off")
    title = f"Tahmin: {res['tur']}" + (f"  ·  Gerçek: {true_label}" if true_label else "")
    ax[0].set_title(title, color="green" if (true_label is None or true_label == res["tur"]) else "red", fontsize=13)
    ax[1].axis("off")
    probs = "  ".join(f"{k}: %{100 * v:.0f}" for k, v in sorted(res["olasiliklar"].items(), key=lambda x: -x[1]))
    fields = "\n".join(f"{k}: {v}" for k, v in res["alanlar"].items())
    txt = f"{res['ozet']}\n\nOlasılıklar: {probs}\nOCR güveni: {res['ocr_guveni']}\n\nAlanlar:\n{fields}"
    ax[1].text(0, 1, txt, va="top", fontsize=10, wrap=True, family="monospace")
    plt.tight_layout(); plt.show()

## 5. RVL-CDIP test belgeleri üzerinde

Modelin eğitimde **görmediği** test belgelerinden her türden ikişer örnek.

In [ ]:
samples = pd.concat([g[~g.is_empty].sample(2, random_state=SEED) for _, g in test.groupby("label")])
for _, r in samples.iterrows():
    res = analyze(load_document(DATA / r.path))
    print(f"[{r.label}] {Path(r.path).name}\n  {res['ozet']}\n")
    show(DATA / r.path, res, true_label=r.label)

## 6. Kendi belgeleriniz

1. Drive'da `bitirme_ocr/kendi_belgelerim/` klasörüne **İngilizce** belgeler koyun (JPG, PNG veya PDF): kendi CV'niz, bir fatura, yazdırılmış bir e-posta, bir form
2. Hücreyi çalıştırın

Kişisel bilgi içeren belgeleri rapora/sunuma koymadan önce maskeleyin.

In [ ]:
files = sorted(p for p in OWN.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".pdf"})
if not files:
    print(f"{OWN} klasörü boş. Belge ekleyip tekrar çalıştırın.")
own_rows = []
for p in files:
    res = analyze(load_document(p))
    own_rows.append({"dosya": p.name, "tahmin": res["tur"], "olasılık": max(res["olasiliklar"].values(), default=0),
                     "özet": res["ozet"]})
    show(p, res)
if own_rows:
    display(pd.DataFrame(own_rows))

## 7. İçerik çıkarımının değerlendirilmesi

Çıkarılan alanların doğruluğu, **türü doğru tahmin edilen** test belgelerinden her türden 5 tane (toplam 20) üzerinde elle değerlendirilir.

Her alan için belgeye bakıp seçin:

| Model bir değer çıkardıysa | Model boş bıraktıysa |
|---|---|
| **doğru** · **yanlış** | **belgede de yok** · **kaçırıldı** |

| Ölçüt | Formül |
|---|---|
| Precision | doğru / (doğru + yanlış) |
| Recall | doğru / (doğru + kaçırıldı) |

Kayıtlar her belgede `cikarim_degerlendirme.csv` dosyasına yazılır; yarıda bırakıp sonra devam edebilirsiniz.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

EVAL_FILE = ROOT / "cikarim_degerlendirme.csv"
pred = model.predict(test.text_norm)
pool = test[(pred == test.label.values) & (~test.is_empty.values)]
eval_docs = pd.concat([g.sample(min(5, len(g)), random_state=SEED) for _, g in pool.groupby("label")])
done = pd.read_csv(EVAL_FILE) if EVAL_FILE.exists() else pd.DataFrame(columns=["belge", "tur", "alan", "deger", "karar"])
todo = [r for _, r in eval_docs.iterrows() if r.path not in set(done.belge)]
print(f"Değerlendirilecek: {len(todo)} belge (tamamlanan: {done.belge.nunique()})")

def fmt(v):
    return ", ".join(map(str, v)) if isinstance(v, list) else (json.dumps(v, ensure_ascii=False) if isinstance(v, dict) else str(v))

out = w.Output()
state = {"i": 0, "widgets": {}, "res": None}

def render():
    with out:
        clear_output(wait=True)
        if state["i"] >= len(todo):
            print("Bitti! Sonuçlar için bir sonraki hücreyi çalıştırın."); return
        r = todo[state["i"]]
        res = analyze(load_document(DATA / r.path)); state["res"] = res
        fig, ax = plt.subplots(figsize=(7, 9)); ax.imshow(load_document(DATA / r.path), cmap="gray"); ax.axis("off")
        ax.set_title(f"{state['i'] + 1}/{len(todo)} · {r.label} · {Path(r.path).name}"); plt.show()
        state["widgets"] = {}
        rows = []
        for k, v in res["alanlar"].items():
            empty = v in (None, [], {})
            opts = ["belgede de yok", "kaçırıldı"] if empty else ["doğru", "yanlış"]
            tb = w.ToggleButtons(options=opts, value=opts[0])
            state["widgets"][k] = (tb, None if empty else fmt(v))
            rows.append(w.HBox([w.HTML(f"<b style='display:inline-block;width:130px'>{k}</b>"),
                                w.HTML(f"<span style='display:inline-block;width:360px'>{'—' if empty else fmt(v)[:80]}</span>"), tb]))
        display(w.VBox(rows))

def save_next(_):
    global done
    if state["i"] >= len(todo): return
    r = todo[state["i"]]
    new = [{"belge": r.path, "tur": r.label, "alan": k, "deger": val, "karar": tb.value}
           for k, (tb, val) in state["widgets"].items()]
    done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
    done.to_csv(EVAL_FILE, index=False)
    state["i"] += 1; render()

btn = w.Button(description="Kaydet ve sonraki →", button_style="success")
btn.on_click(save_next)
display(btn, out); render()

In [ ]:
ev = pd.read_csv(EVAL_FILE) if EVAL_FILE.exists() else pd.DataFrame()
if ev.empty:
    print("Henüz değerlendirme yok.")
else:
    def scores(g):
        c = g.karar.value_counts()
        d, y, k = c.get("doğru", 0), c.get("yanlış", 0), c.get("kaçırıldı", 0)
        return pd.Series({"doğru": d, "yanlış": y, "kaçırıldı": k,
                          "precision": d / (d + y) if d + y else np.nan, "recall": d / (d + k) if d + k else np.nan})
    by_field = ev.groupby(["tur", "alan"]).apply(scores).round(3)
    by_type = ev.groupby("tur").apply(scores).round(3)
    display(by_type); display(by_field)
    by_field.to_csv(ROOT / "cikarim_sonuclari.csv")
    print(f"Değerlendirilen belge: {ev.belge.nunique()}")

## 8. Demo arayüzü (Gradio)

Hücre çalışınca bir **public link** (`https://....gradio.live`) çıkar. Bu linki tarayıcıda açıp belge yükleyebilir, hocanıza da gönderebilirsiniz.
Link Colab oturumu açık kaldığı sürece (en fazla ~72 saat) çalışır.

In [ ]:
import gradio as gr

import traceback

def demo_fn(file):
    if file is None:
        return {}, "Bir belge yükleyin.", {}, ""
    try:
        path = file if isinstance(file, str) else getattr(file, "name", file)
        res = analyze(load_document(path))
        probs = {TR_NAMES.get(k, k): float(v) for k, v in res["olasiliklar"].items()}
        fields = json.loads(json.dumps(res["alanlar"], ensure_ascii=False, default=str))
        return probs, f"### {res['ozet']}", fields, res["ocr_metni"]
    except Exception as e:
        tb = traceback.format_exc()
        print(tb)                                   # Colab hücre çıktısına da yazılır
        return {}, f"**Hata:** `{type(e).__name__}: {e}`\n\n```\n{tb[-1500:]}\n```", {}, ""

# Önce Gradio olmadan bir test: kendi_belgelerim'deki ilk dosya
_first = next((p for p in sorted(OWN.iterdir()) if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".pdf"}), None)
if _first:
    print("Test:", _first.name, "→", demo_fn(str(_first))[1][:300])

app = gr.Interface(
    fn=demo_fn,
    inputs=gr.File(label="Belge (JPG, PNG veya PDF)", file_types=["image", ".pdf"]),
    outputs=[gr.Label(num_top_classes=4, label="Belge türü"), gr.Markdown(label="Özet"),
             gr.JSON(label="Çıkarılan alanlar"), gr.Textbox(label="OCR metni", lines=10)],
    title="OCR Tabanlı Belge Sınıflandırma ve İçerik Çıkarımı",
    description="E-posta · Form · Fatura · CV — İngilizce belgeler için. OCR: Tesseract · Sınıflandırma: TF-IDF + " + info["model"],
    flagging_mode="never",
)
app.launch(share=True, debug=False, show_error=True)

## Çıktılar

| Dosya | İçerik |
|---|---|
| `models/demo_model.joblib` | Olasılık veren model |
| `extract.py` | İçerik çıkarım kuralları |
| `cikarim_degerlendirme.csv`, `cikarim_sonuclari.csv` | Elle değerlendirme ve alan bazında precision / recall |